In [1]:
import numpy as np
import pandas as pd

# ============================================================
# 0. Load raw KPI dataset
# ============================================================

data_path = "../data/processed/compustat_kpis.csv"
df = pd.read_csv(data_path)

print("Original shape:", df.shape)
display(df.head())

# ============================================================
# 1. Filter using dlrsn
#    - Keep NaN (still listed)
#    - Keep 2 or 3 (bankruptcy delisting)
#    - Drop all other values
# ============================================================

if "dlrsn" in df.columns:
    print("\nRaw dlrsn counts:")
    print(df["dlrsn"].value_counts(dropna=False))

    df = df[df["dlrsn"].isna() | df["dlrsn"].isin([2, 3])]
    print("\nShape after keeping only dlrsn = NaN, 2, 3:", df.shape)
else:
    print("\n⚠️ dlrsn column not found — skipping filtering.")

# ============================================================
# 1bis. Ensure dlrsn = 2 or 3 appears ONLY in last year per gvkey
# ============================================================

if "dlrsn" in df.columns and "fyear" in df.columns and "gvkey" in df.columns:

    # Sort values properly
    df = df.sort_values(["gvkey", "fyear"])

    # Identify last year per firm
    last_year = df.groupby("gvkey")["fyear"].transform("max")

    # For each row:
    # If dlrsn in [2,3] AND NOT last year → remove bankruptcy tag
    mask_wrong_year = (df["dlrsn"].isin([2, 3])) & (df["fyear"] != last_year)

    # Set dlrsn to NaN for those years
    df.loc[mask_wrong_year, "dlrsn"] = np.nan

    print("\nApplied rule: Bankruptcy code kept only for last available year.")
    print(df["dlrsn"].value_counts(dropna=False))

else:
    print("\n⚠️ Missing dlrsn or fyear or gvkey — cannot apply last-year bankruptcy rule.")

# ============================================================
# 2. Select only ID columns + KPI columns + KEEP dlrsn
# ============================================================

id_cols = ["gvkey", "datadate", "fyear", "conm", "tic", "dlrsn"]
id_cols = [c for c in id_cols if c in df.columns]

kpi_cols = [
    "roa",
    "ebitda_margin",
    "debt_ratio",
    "total_debt_to_equity",
    "current_ratio",
    "cash_ratio",
    "interest_coverage",
    "cfo_margin",
    "free_cash_flow",
    "asset_turnover",
    "sales_growth",
    "asset_growth",
    "book_to_market",
    "price_to_book",
    "working_capital_to_assets",
    "retained_earnings_to_assets",
]

kpi_cols = [c for c in kpi_cols if c in df.columns]

print("\nKPI columns kept:", kpi_cols)
print("Number of KPIs:", len(kpi_cols))

# Keep ALL IDs + ALL KPIs + dlrsn
df = df[id_cols + kpi_cols]

print("Shape after column selection:", df.shape)
display(df.head())

# ============================================================
# 3. Basic cleaning
# ============================================================

# Replace infinities
df[kpi_cols] = df[kpi_cols].replace([np.inf, -np.inf], np.nan)

# Drop rows where all KPIs are missing
df = df.dropna(subset=kpi_cols, how="all")
print("Shape after dropping rows with all KPI = NaN:", df.shape)

# Drop rows with >50% missing KPIs
row_missing_ratio = df[kpi_cols].isna().mean(axis=1)
df = df[row_missing_ratio <= 0.5]
print("Shape after dropping rows with >50% missing KPIs:", df.shape)

# ============================================================
# 4. Fill missing values using the median
# ============================================================

for col in kpi_cols:
    df[col] = df[col].fillna(df[col].median())

print("Remaining NaN count:", df[kpi_cols].isna().sum().sum())

# ============================================================
# 5. Winsorization (1% - 99%)
# ============================================================

for col in kpi_cols:
    low = df[col].quantile(0.01)
    high = df[col].quantile(0.99)
    df[col] = df[col].clip(low, high)

print("Winsorization applied.")

# ============================================================
# 6. Save CLEAN (unscaled) dataset — KEEP dlrsn
# ============================================================

output_clean = "../data/processed/compustat_kpis_clean.csv"
df.to_csv(output_clean, index=False)
print("Clean file saved to:", output_clean)

# ============================================================
# 7. Standardize KPI columns (z-score) — KEEP dlrsn unchanged
# ============================================================

df_scaled = df.copy()

for col in kpi_cols:
    mean = df_scaled[col].mean()
    std = df_scaled[col].std()
    df_scaled[col] = 0 if (std == 0 or np.isnan(std)) else (df_scaled[col] - mean) / std

print("Standardization done.")

# ============================================================
# 8. Save scaled dataset — KEEP dlrsn
# ============================================================

output_scaled = "../data/processed/compustat_kpis_clean_scaled.csv"
df_scaled.to_csv(output_scaled, index=False)
print("Scaled file saved to:", output_scaled)

# ============================================================
# 9. Summary statistics (only on KPI columns)
# ============================================================

summary = df[kpi_cols].describe().T
summary_path = "../data/processed/kpi_summary_stats.csv"
summary.to_csv(summary_path)
print("Summary statistics saved to:", summary_path)

print("\nFinal shapes:")
print("Clean (unscaled):", df.shape)
print("Clean + scaled:", df_scaled.shape)
display(summary.head())


Original shape: (332675, 57)


,gvkey,datadate,fyear,conm,tic,act,lct,at,lt,seq,...,interest_coverage,cfo_margin,free_cash_flow,asset_turnover,sales_growth,asset_growth,working_capital_to_assets,retained_earnings_to_assets,book_to_market,price_to_book
0,1004,1995-05-31,1994,AAR CORP,AIR,321.632,73.140,425.814,228.695,197.119,...,2.242018,0.033795,6.182,1.060076,NaN,NaN,0.583569,0.240565,NaN,1.234818
1,1004,1996-05-31,1995,AAR CORP,AIR,338.012,79.385,437.846,233.211,204.635,...,3.055953,0.049031,17.213,1.153351,0.118732,0.028256,0.590680,0.250182,NaN,1.729691
2,1004,1997-05-31,1996,AAR CORP,AIR,414.100,99.981,529.584,260.325,269.259,...,3.976451,0.016173,-20.761,1.112813,0.167009,0.209521,0.593143,0.231646,NaN,2.095840
3,1004,1998-05-31,1997,AAR CORP,AIR,468.400,149.148,670.559,369.709,300.850,...,4.465020,0.029181,5.328,1.166375,0.327144,0.266200,0.476098,0.220100,NaN,2.434527
4,1004,1999-05-31,1998,AAR CORP,AIR,508.186,173.586,726.630,400.595,326.035,...,4.167663,0.031072,-7.606,1.263416,0.173774,0.083618,0.460482,0.245524,0.602903,1.658646



Raw dlrsn counts:
dlrsn
NaN     171076
1.0      97570
10.0     34180
7.0      11066
3.0       8435
4.0       4296
2.0       3689
9.0       2165
5.0        198
Name: count, dtype: int64

Shape after keeping only dlrsn = NaN, 2, 3: (183200, 57)

Applied rule: Bankruptcy code kept only for last available year.
dlrsn
NaN    181115
3.0      1663
2.0       422
Name: count, dtype: int64

KPI columns kept: ['roa', 'ebitda_margin', 'debt_ratio', 'total_debt_to_equity', 'current_ratio', 'cash_ratio', 'interest_coverage', 'cfo_margin', 'free_cash_flow', 'asset_turnover', 'sales_growth', 'asset_growth', 'book_to_market', 'price_to_book', 'working_capital_to_assets', 'retained_earnings_to_assets']
Number of KPIs: 16
Shape after column selection: (183200, 22)


,gvkey,datadate,fyear,conm,tic,dlrsn,roa,ebitda_margin,debt_ratio,total_debt_to_equity,...,interest_coverage,cfo_margin,free_cash_flow,asset_turnover,sales_growth,asset_growth,book_to_market,price_to_book,working_capital_to_assets,retained_earnings_to_assets
0,1004,1995-05-31,1994,AAR CORP,AIR,NaN,0.031793,0.077019,0.537077,0.615861,...,2.242018,0.033795,6.182,1.060076,NaN,NaN,NaN,1.234818,0.583569,0.240565
1,1004,1996-05-31,1995,AAR CORP,AIR,NaN,0.049849,0.084273,0.532632,0.585266,...,3.055953,0.049031,17.213,1.153351,0.118732,0.028256,NaN,1.729691,0.590680,0.250182
2,1004,1997-05-31,1996,AAR CORP,AIR,NaN,0.060621,0.093627,0.491565,0.439324,...,3.976451,0.016173,-20.761,1.112813,0.167009,0.209521,NaN,2.095840,0.593143,0.231646
3,1004,1998-05-31,1997,AAR CORP,AIR,NaN,0.074896,0.101006,0.551344,0.590813,...,4.465020,0.029181,5.328,1.166375,0.327144,0.266200,NaN,2.434527,0.476098,0.220100
4,1004,1999-05-31,1998,AAR CORP,AIR,NaN,0.080941,0.102876,0.551305,0.556256,...,4.167663,0.031072,-7.606,1.263416,0.173774,0.083618,0.602903,1.658646,0.460482,0.245524


Shape after dropping rows with all KPI = NaN: (140685, 22)
Shape after dropping rows with >50% missing KPIs: (121916, 22)
Remaining NaN count: 0
Winsorization applied.
Clean file saved to: ../data/processed/compustat_kpis_clean.csv
Standardization done.
Scaled file saved to: ../data/processed/compustat_kpis_clean_scaled.csv
Summary statistics saved to: ../data/processed/kpi_summary_stats.csv

Final shapes:
Clean (unscaled): (121916, 22)
Clean + scaled: (121916, 22)


,count,mean,std,min,25%,50%,75%,max
roa,121916.0,-0.295502,1.514095,-12.392984,-0.051513,0.025405,0.068334,0.366544
ebitda_margin,121916.0,-1.530499,9.606023,-81.330702,0.036082,0.132535,0.262280,0.818166
debt_ratio,121916.0,0.931057,2.218173,0.027338,0.363083,0.591298,0.828219,19.463228
total_debt_to_equity,121916.0,0.813893,2.628242,-9.588279,0.012543,0.380569,1.054195,16.184689
current_ratio,121916.0,2.713269,3.658335,0.014123,1.160696,1.698897,2.587770,25.176805
